In [0]:
# SECTION 1 — SECRET SCOPE
SECRET_SCOPE = "adls-scope"
SECRET_CLIENT_ID     = "app-client-id"
SECRET_CLIENT_SECRET = "app-client-secret"
SECRET_TENANT_ID     = "app-tenant-id"

In [0]:
# SECTION 2 — STORAGE ACCOUNT
STORAGE_ACCOUNT = "asadev"
CONTAINER_NAME = "kyc-aml-dev"
# abfss://"kyc-aml-dev"@"asadev.dfs.core.windows.net/ecommerce-sandbox-operations.dynamics.com/Tables/Customers/olist_customers_dataset.csv"

In [0]:
# SECTION 3 - BASE PATH
BASE_PATH = f"abfss://{CONTAINER_NAME}@{STORAGE_ACCOUNT}.dfs.core.windows.net"
TABLE_BASE_PATH = f"{BASE_PATH}/ecommerce-sandbox-operations.dynamics.com/Tables"


In [0]:
# SECTION 4 - PATH FOR ALL CSV FILES
RAW_PATHS = {
    "customers"           : f"{TABLE_BASE_PATH}/Customers/olist_customers_dataset.csv",
    "geolocation"         : f"{TABLE_BASE_PATH}/GeoLocation/olist_geolocation_dataset.csv",
    "orders"              : f"{TABLE_BASE_PATH}/Orders/olist_orders_dataset.csv",
    "order_items"         : f"{TABLE_BASE_PATH}/Orders/olist_order_items_dataset.csv",
    "order_payments"      : f"{TABLE_BASE_PATH}/Orders/olist_order_payments_dataset.csv",
    "order_reviews"       : f"{TABLE_BASE_PATH}/Orders/olist_order_reviews_dataset.csv",
    "products"            : f"{TABLE_BASE_PATH}/Purchase/olist_products_dataset.csv",
    "category_translation": f"{TABLE_BASE_PATH}/Purchase/product_category_name_translation.csv",
    "sellers"             : f"{TABLE_BASE_PATH}/Sales/olist_sellers_dataset.csv",
}


In [0]:
# SECTION 5 - UNITY CATALOG
CATALOG_NAME = "fincompliance_catalog"

BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"
GOLD_SCHEMA = "gold"

BRONZE_DB = f"{CATALOG_NAME}.{BRONZE_SCHEMA}"
SILVER_DB = f"{CATALOG_NAME}.{SILVER_SCHEMA}"
GOLD_DB = f"{CATALOG_NAME}.{GOLD_SCHEMA}"

In [0]:

# SECTION 6 - Authentication Function
def authenticate_spark(spark):
    client_id = dbutils.secrets.get(scope = SECRET_SCOPE, key = SECRET_CLIENT_ID)
    client_secret = dbutils.secrets.get(scope = SECRET_SCOPE, key = SECRET_CLIENT_SECRET)
    tenant_id = dbutils.secrets.get(scope = SECRET_SCOPE, key =SECRET_TENANT_ID  )

    spark.conf.set(
        f"fs.azure.account.auth.type.{STORAGE_ACCOUNT}.dfs.core.windows.net",
        "OAuth"
    )
    spark.conf.set(
        f"fs.azure.account.oauth.provider.type.{STORAGE_ACCOUNT}.dfs.core.windows.net",
        "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider"
    )

    spark.conf.set(
        f"fs.azure.account.oauth2.client.id.{STORAGE_ACCOUNT}.dfs.core.windows.net",
        client_id
    )

    spark.conf.set(
        f"fs.azure.account.oauth2.client.secret.{STORAGE_ACCOUNT}.dfs.core.windows.net",
        client_secret
    )
    spark.conf.set(
        f"fs.azure.account.oauth2.client.endpoint.{STORAGE_ACCOUNT}.dfs.core.windows.net",
        f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
    )

    print(f"Spark authenticated to {STORAGE_ACCOUNT} successfully")


In [0]:
# SECTION 7 - SET CATALOG FUNCTION
def set_catalog(spark):
    spark.catalog.setCurrentCatalog(CATALOG_NAME)
    print(f"Catalog set to {CATALOG_NAME}")

In [0]:
# SECTION 8 - VERIFY CONFIG
if __name__ == "__main__":
    print("=" * 60)
    print("CONFIG VERIFICATION")
    print("=" * 60)
    print(f"\nSecret Scope     : {SECRET_SCOPE}")
    print(f"Storage Account  : {STORAGE_ACCOUNT}")
    print(f"Container        : {CONTAINER_NAME}")
    print(f"Catalog          : {CATALOG_NAME}")
    print(f"Bronze Schema    : {BRONZE_DB}")
    print(f"Silver Schema    : {SILVER_DB}")
    print(f"Gold Schema      : {GOLD_DB}")
    print(f"\nRAW FILE PATHS:")
    for table, path in RAW_PATHS.items():
        print(f"  {table:<25} : {path}")
    print("\n Config loaded successfully")

In [0]:
print("="*60)
# TESTING THE AUTHENTICATION ACCESS 
print("="*60)

print("Step 1: Testing authentication...")
authenticate_spark(spark)

print("Step 2: Testing catalog")
set_catalog(spark)

print("\nSTEP 3: Testing ADLS file access")

for table_name, path in RAW_PATHS.items():
    try:
        files = dbutils.fs.ls(path)
        print(f"{table_name:<25} : filefound")
    except Exception as e:
        print(f"Error accessing {table_name:<25} : {e}")

print("\n Step 4 : Testing whether spark can read the file")
try:
    df = spark.read \
        .option("header", "true")\
        .csv(RAW_PATHS["customers"])
    print(f" Customers file readable - {df.count()} rows found")
    print(f" Columns: {df.columns}")
except Exception as e:
    print(f"Could not read the file {e}")